In [ ]:
# --- Colab bootstrap: in Colab, clone + install the repo; locally, do nothing ---
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/CKmellow/per3-genotype-clock-gene-timing.git"
BRANCH = "main"  # change to your branch name to test it in Colab
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    repo = Path("/content/per3-genotype-clock-gene-timing")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(repo)], check=True)
    os.chdir(repo)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
    src = str(repo / "src")
else:  # local: find the repo root (works from notebooks/ or notebooks/stages/)
    here = Path.cwd().resolve()
    root = next((p for p in [here, *here.parents] if (p / "pyproject.toml").exists()), here)
    src = str(root / "src")
if src not in sys.path:
    sys.path.insert(0, src)  # makes `import per3clock` work without a kernel restart

from per3clock import config  # noqa: E402

config.set_seeds()
config.ensure_dirs()
print("per3clock root:", config.ROOT, "| Colab:", IN_COLAB)

# Stage 3 · Preprocessing and quality control

*Owner: Part 2 (see issue #2).*

## Minimum evidence (lecturer's brief)
*Brief, stage 3: "Clean, filter, normalize or align the data; check it against biology you know must hold." Minimum evidence: "Before/after counts, a comment on anything suspicious."*

- [ ] Clean / filter / normalize: the data are already quantile-normalised by GEO (limma). Show that array distributions are comparable, and document every filter you add
- [ ] **Before/after counts** for every step (samples, subjects per genotype, probes/genes)
- [ ] **Check against biology you know must hold** (e.g. core clock genes expressed above background in blood, which matters most for PER3, the weakest; clock genes vary with time of day in the expected order; sex-marker genes consistent within each subject)
- [ ] **A comment on anything suspicious** (outlier arrays, AF0182 SE phase labels, recovered samples, late draws)
- [ ] Review the shared defaults (`analysis_samples()`, one-probe-per-gene) and say whether QC supports them
- [ ] Short interpretation of your own results
- [ ] Report what the data does **and does not** show. A null, negative or inconclusive result is a valid result: report it with its uncertainty.

## Contract
**Inputs:** `data.load_metadata()`, `data.load_expression_matrix()`, `data.load_probe_annotation()`, `data.collapse_probes_to_genes()` (default: one probe per gene, highest mean), `data.analysis_samples(..., return_exclusions=True)`, `results/tables/selected_probes.csv`, `sample_anomalies.csv`, `timepoints_per_subject.csv`.

**Shared defaults (issue #10, `data/README.md` → *Shared defaults*):** main set = SE, ≥6 timepoints (24 subjects, 221 samples). `strict=True` = authors' rule (20 subjects, sensitivity). `for_phase_analysis=True` drops AF0182 SE (23 subjects). These are the single source of truth: if QC justifies a change, change `analysis_samples()` in its own PR (tag Part 1) rather than saving a separate sample list.

**Implement in `per3clock.qc`:** `filter_low_expression`, `sample_qc_table`, `apply_qc`, `qc_counts`. Remove the `xfail` markers in `tests/test_qc.py` once done.

**Outputs (names are fixed):**
- `results/tables/qc_counts.csv`: before/after counts per step
- `results/tables/qc_exclusions.csv`: `sample_id, subject, sleep_condition, reason`
- figures: `qc_expression_density.png`, `qc_sample_correlation.png`, `qc_biology_checks.png` (via `plotting.save_fig`, 300 dpi)
- extra tables: `qc_filter_sensitivity.csv`, `qc_sample_metrics.csv`, `qc_clock_gene_background.csv`, `qc_clock_timing_pooled.csv`, `qc_sex_check.csv`
- `data/processed/expressed_genes_SE.csv`: the expressed-gene list (feature set and enrichment background for Parts 3–4)

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

from per3clock import config, data, plotting, qc

plotting.set_style()
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
FIG_DPI = 300  # report figures

## 3.1 Load data

Everything comes through `per3clock.data`. The main analysis is **Sleep Extension only** (`analysis_samples()`: 24 subjects, 221 samples). Sleep Restriction is a sensitivity analysis and is never mixed into the main QC.

Before any transformation we check what the matrix already is. GEO says the data were quantile-normalised with limma; if that is true every sample has the same distribution, and re-normalising or re-logging would distort it.

In [ ]:
meta = data.load_metadata()
annot = data.load_probe_annotation()
expr = data.load_expression_matrix()            # probes x 438 samples (downloaded once, ~100 MB)
main = data.analysis_samples(metadata=meta)     # shared default: SE, >= 6 timepoints

gt = main.drop_duplicates("subject").per3_genotype.value_counts()
print(f"series matrix: {expr.shape[0]:,} probes x {expr.shape[1]} samples")
print(f"main set: {len(main)} samples, {main.subject.nunique()} subjects "
      f"({gt['4/4']} x 4/4, {gt['5/5']} x 5/5), conditions: {sorted(main.sleep_condition.unique())}")

chk = qc.check_log2_normalised(expr)
print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in chk.items() if k != "quantile_spread"})
display(chk["quantile_spread"].to_frame("max - min across 438 samples").T.round(4))

**Result.** Values run from about 1.1 to 19.5 with median 6.70 and no missing cells, which is a log2 scale (a linear Agilent signal spans ~10⁰–10⁶). Every sample's 5th, 25th, 50th, 75th and 95th percentile agrees to within 0.0004 log2 units: the matrix is already quantile-normalised. **We apply no further transformation or normalisation.**

## 3.2 Probe/gene-level filtering

**Steps.** (1) Keep probes with a gene symbol. (2) One probe per gene, the shared outcome-blind default (`collapse_probes_to_genes`, highest mean over all 438 samples, issue #10; not re-selected here). (3) Remove genes that are not expressed above the array background.

**Threshold rule (chosen before looking at any genotype or time effect).** The array carries 757 negative-control spots (153 `(-)3xSLv` and 604 `DarkCorner`) that measure signal with no transcript. Following the limma user guide's Agilent example, a value counts as expressed if it is at least 10 % brighter than the 95th percentile of the negative controls, i.e. `threshold = Q95(negative controls in the 221 main samples) + log2(1.1)`. A gene is kept if it is above the threshold in **≥ 20 % of the main samples**: about 2 of a subject's 10 three-hourly samples, so a gene expressed only around its daily peak is not removed, and well below the smaller genotype group's share (~45 %), so a gene expressed in one genotype only cannot be filtered out. The rule uses the expression values only, never time, genotype or condition.

The placeholder in the stub (log2 > 6 in 20 % of samples) was **not** used: it would remove PER3, a primary gene (see the sensitivity table).

In [ ]:
has_symbol = annot.GENE_SYMBOL.reindex(expr.index).notna()
genes_all = data.collapse_probes_to_genes(expr, annot)       # one probe per gene, all 438 samples
genes_main = genes_all[main.sample_id]

neg_ids = qc.negative_control_ids(annot).intersection(expr.index)
threshold = qc.background_threshold(expr[main.sample_id], annot)
expressed = qc.filter_low_expression(genes_main, threshold, qc.DEFAULT_MIN_FRAC)

print(f"negative-control probes: {len(neg_ids)}; threshold = log2 {threshold:.3f}")
print(f"probes {len(expr):,} -> with gene symbol {int(has_symbol.sum()):,} -> one per gene "
      f"{len(genes_all):,} -> expressed {len(expressed):,} genes "
      f"({len(genes_all) - len(expressed):,} removed)")

sens = qc.filter_sensitivity(genes_main, min_log2_values=(2.5, threshold, 3.5, 4.0, 5.0, 6.0))
display(sens)

fig = qc.plot_expression_qc(genes_main, main, threshold=threshold,
                            negative_controls=expr.loc[neg_ids, main.sample_id])
plotting.save_fig(fig, "qc_expression_density", dpi=FIG_DPI)

In [ ]:
# All main-analysis clock genes must survive the filter
lost = [g for g in config.CLOCK_GENES if g not in expressed.index]
print("clock genes removed by the filter:", lost or "none")
assert not lost, lost
selected = pd.read_csv(config.TABLES_DIR / "selected_probes.csv")
display(selected[selected.selected][["gene", "probe_id", "probe_name", "mean_log2"]])

**Figure `qc_expression_density`.** (a) The per-sample boxes are identical, which is what quantile normalisation should give, so there is no array with a shifted or compressed distribution. (b) The negative-control signal (black) forms a narrow peak near log2 1.5–2.5. Gene means have a small unexpressed component overlapping it and a broad expressed component centred near 8–9. The threshold sits at the upper edge of the background peak. PER3, the weakest primary gene, has a mean of 5.5: well above background, though below the array median.

## 3.3 Sample-level QC

Metrics per sample, computed on the expressed genes (`qc.sample_qc_table`):
- median and IQR (a check of the normalisation) and missing values;
- Pearson r with the median profile of the **same subject's other samples** (leave-one-out). Between-subject differences are large in blood, so this is the sensitive test for a failed or swapped array;
- Pearson r with the median of all samples;
- the 95th percentile of the negative controls (array background), reported but not used to flag.

**Outlier rule:** a sample is flagged if either correlation, after Fisher's z transform, is more than 3 robust SDs (1.4826 × MAD) below the median of all samples. Flagged samples are **reported, not removed**: dropping samples beyond the agreed sets would need a group decision.

In [ ]:
background = qc.negative_control_p95(expr[main.sample_id], annot)
sample_qc = qc.sample_qc_table(expressed, main, background=background)
metric_cols = ["median_log2", "iqr_log2", "n_missing", "corr_to_subject_median",
               "corr_to_all_median", "neg_ctrl_p95"]
display(sample_qc[metric_cols].describe().T.round(4))

show = ["subject", "per3_genotype", "timepoint", "clock_time", "corr_to_subject_median",
        "corr_to_all_median", "z_subject", "z_all", "neg_ctrl_p95", "outlier_reason"]
print(f"Flagged outliers: {int(sample_qc.outlier.sum())} of {len(sample_qc)}")
display(sample_qc.loc[sample_qc.outlier, show].round(3))
print(f"High-background arrays (negative-control p95, robust z > 3): {int(sample_qc.high_background.sum())}")
display(sample_qc.loc[sample_qc.high_background, show].round(3))

fig = qc.plot_sample_qc(sample_qc)
plotting.save_fig(fig, "qc_sample_correlation", dpi=FIG_DPI)

In [ ]:
# Samples already known to be unusual (data/README.md, Known issues): how do they look in QC?
anomalies = pd.read_csv(config.TABLES_DIR / "sample_anomalies.csv")
recovered = main.loc[main.is_recovered, "sample_id"]
known = pd.concat([anomalies[["sample_id", "issue"]],
                   pd.DataFrame({"sample_id": recovered, "issue": "subject/genotype recovered from title"})])
known = known[known.sample_id.isin(sample_qc.index)]
display(known.merge(sample_qc[show[:4] + ["corr_to_subject_median", "z_subject", "outlier"]],
                    left_on="sample_id", right_index=True).round(3))

**Figure `qc_sample_correlation` and tables.** No array looks failed or swapped. Every sample correlates strongly with its own subject's other samples (median r 0.993, minimum 0.969) and with the all-sample median (minimum 0.962); medians and IQRs are identical across samples and there are no missing values.

- **4 mild statistical outliers** (robust z < −3 on the within-subject correlation): AF0069 T10 (r = 0.969), AF0114 T1 and T6 (0.974 each) and AF0145 T4 (0.972). They form the lower tail of a tight distribution rather than standing apart from it, and their correlation with the all-sample median is ordinary. **Kept** (no agreed rule to drop them); listed in `qc_sample_metrics.csv`. *Proposal for the group, not applied:* Part 3 could check that per-subject fits for AF0069, AF0114 and AF0145 do not change materially without these samples (`qc.apply_qc(meta, expr, drop_outliers=True)`); no subject would fall below 6 timepoints.
- **6 arrays with bright negative-control spots** (95th percentile 4.9–6.3 vs median 2.4): AF0101 T1, AF0186 T5, AF0191 T1 and T2, AF0295 T2, AF0335 T5. Their gene-level correlations are normal (0.979–0.994), so this looks like a local artefact at the control spots rather than a poor hybridisation. Reported, not acted on.
- **Known issues look normal at the expression level:** AF0182 SE T4/T5 (|z| ≤ 1.1), the late draws AF0186 T8 and AF0318 T10, and the 11 recovered samples all have ordinary metrics. AF0182's problem is in its phase labels, not its arrays, and the recovered samples behave like their subject's other samples, which supports the recovery of their subject IDs.

## 3.4 Time-series completeness

The shared sample sets (`analysis_samples`, issue #10) are applied unchanged through `qc.apply_qc`, which returns the kept samples and a per-sample exclusion table with reasons.

In [ ]:
variants = {
    "default (main)": {},
    "strict=True (authors' rule, sensitivity)": {"strict": True},
    "for_phase_analysis=True": {"for_phase_analysis": True},
    "Sleep Restriction (sensitivity)": {"condition": "Sleep Restriction"},
}
sets = {}
for label, kw in variants.items():
    kept, excl = qc.apply_qc(meta, **kw)
    sets[label] = kept
    cond = kw.get("condition", config.MAIN_CONDITION)
    print(f"--- {label}: {len(kept)} samples, {kept.subject.nunique()} subjects")
    display(excl[excl.sleep_condition == cond].groupby(["subject", "reason"]).size()
            .rename("samples excluded").reset_index())

kept_main, exclusions = qc.apply_qc(meta)
assert list(kept_main.sample_id) == list(main.sample_id)   # QC keeps exactly the agreed main set
print("Main-set exclusions by reason:")
display(exclusions.reason.value_counts().to_frame("samples"))

Every subject with Sleep Extension samples already has ≥ 6 SE timepoints, so the default removes no SE sample: all 217 exclusions from the main set are Sleep Restriction samples, set aside by design (AF0079 and AF0091 have only SR samples, which is why the main set has 24 subjects). The authors' stricter rule removes AF0112, AF0182, AF0186 and AF0267 (20 subjects, 193 samples). AF0182 SE stays in clock-time analyses and is removed only when time is `circadian_phase_h` (23 subjects, 214 samples): its T4 (02:25) and T5 (04:25) share phase −3 h, and T4 implies phase 0 two hours earlier than its other samples. Its expression is unremarkable in 3.3, which supports a labelling error rather than a bad array.

## 3.5 Biology sanity checks

Things that must hold if the data and labels are right:
1. **Core clock genes are expressed in blood**, above background. This matters most for PER3, the weakest primary gene on this array.
2. **Clock genes vary with time of day in the expected order**: the repressor arm (PER1–3, NR1D1/2) peaks together, roughly opposite the activator ARNTL. Checked with one cosinor per gene on subject-centred expression, genotypes pooled (`qc.pooled_clock_timing`). This is a description of the data, not the per-subject analysis (Part 3), and no p-value is given because pooled repeated measures are not independent.
3. **Sex markers are consistent within each subject.** Sex is not in the GEO metadata; XIST (expressed from the inactive X) and RPS4Y1 (Y chromosome) are used only to check that no subject's samples disagree, which would suggest a sample swap.

In [ ]:
clock = data.load_clock_genes()                 # one probe per gene (shared default)
clock_main = clock[clock.sample_id.isin(main.sample_id)]

bg_check = qc.clock_gene_background_check(genes_main, threshold)
display(bg_check.round(3))

timing = qc.pooled_clock_timing(clock_main)
timing["acrophase_clock"] = timing.acrophase_h.map(lambda h: f"{int(h):02d}:{int(round(h % 1 * 60)) % 60:02d}")
display(timing.round(3))

sex = qc.infer_sex(genes_main, main)
per_subject = sex.drop_duplicates("subject")[["subject", "per3_genotype", "subject_sex"]]
print("samples inconsistent with their subject:", int((~sex.consistent).sum()))
display(pd.crosstab(per_subject.per3_genotype, per_subject.subject_sex))

fig = qc.plot_biology_checks(clock_main, timing, sex, threshold=threshold, array_median=chk["median"])
plotting.save_fig(fig, "qc_biology_checks", dpi=FIG_DPI)

**Figure `qc_biology_checks`.**

1. **Above background.** All eight clock genes exceed the threshold in 100 % of main samples. PER3 is the weakest (mean log2 5.46, 30th percentile of all genes, minimum 4.21 against a threshold of 3.04): low but clearly expressed, so its rhythm estimates will be noisier than PER2's or NR1D1's, not invalid. CLOCK (34th percentile) and CRY1 (38th) are also low.
2. **Expected order.** Pooled peak times: NR1D1 02:13, NR1D2 04:09, PER2 05:03, PER3 06:03, PER1 06:52, so the repressor arm peaks together in the early morning, NR1D1 first. ARNTL's pooled rhythm is weak (r² 0.03), but its peak (14:57) is roughly opposite the PER/NR1D genes, as the clock's feedback loop predicts. CLOCK (r² 0.00) and CRY1 (r² 0.03) show essentially no pooled rhythm, which supports treating them as secondary. These are pooled descriptions only; Part 3 estimates per-subject peaks with uncertainty.
3. **Sex markers.** Every sample matches its subject (0 inconsistent): 10 XIST-high and 14 RPS4Y1-high subjects, with no sign of a sample swap. The split is 5/6 in the 4/4 group and 5/8 in the 5/5 group, roughly balanced; Part 3 may still want sex as a covariate or check.

## 3.6 Before/after table and outputs

In [ ]:
n_probes, n_symbol, n_genes, n_expr = len(expr), int(has_symbol.sum()), len(genes_all), len(expressed)
counts = qc.qc_counts([
    ("1 GEO series matrix (all samples, all probes)", meta, n_probes),
    ("2 probes with a gene symbol", meta, n_symbol),
    ("3 one probe per gene (shared default)", meta, n_genes),
    ("4 Sleep Extension, >=6 timepoints (main set)", main, n_genes),
    ("5 expression filter (main set)", main, n_expr),
    ("6 sample outliers removed (none: flagged only)", kept_main, n_expr),
    ("sensitivity: strict=True", sets["strict=True (authors' rule, sensitivity)"], n_expr),
    ("sensitivity: for_phase_analysis=True", sets["for_phase_analysis=True"], n_expr),
    ("sensitivity: Sleep Restriction (SE gene set)", sets["Sleep Restriction (sensitivity)"], n_expr),
])
display(counts)

counts.to_csv(config.TABLES_DIR / "qc_counts.csv", index=False)
exclusions.to_csv(config.TABLES_DIR / "qc_exclusions.csv", index=False)
sens.to_csv(config.TABLES_DIR / "qc_filter_sensitivity.csv", index=False)
sample_qc.drop(columns=["title"]).round(5).to_csv(config.TABLES_DIR / "qc_sample_metrics.csv")
bg_check.round(4).to_csv(config.TABLES_DIR / "qc_clock_gene_background.csv")
timing.round(4).to_csv(config.TABLES_DIR / "qc_clock_timing_pooled.csv")
sex.round(3).to_csv(config.TABLES_DIR / "qc_sex_check.csv")

sel = data.select_probes(expr, annot).loc[expressed.index]
expressed_table = pd.DataFrame({
    "gene": expressed.index, "probe_id": sel.probe_id.to_numpy(),
    "mean_log2_SE": expressed.mean(axis=1).round(4).to_numpy(),
    "frac_above_threshold": (expressed > threshold).mean(axis=1).round(4).to_numpy(),
})
expressed_table.to_csv(config.PROCESSED_DIR / "expressed_genes_SE.csv", index=False)
print("saved:", ", ".join(["qc_counts.csv", "qc_exclusions.csv", "qc_filter_sensitivity.csv",
      "qc_sample_metrics.csv", "qc_clock_gene_background.csv", "qc_clock_timing_pooled.csv",
      "qc_sex_check.csv", "data/processed/expressed_genes_SE.csv"]))

## Interpretation

**What was changed, and why (counts in `qc_counts.csv`).**
- *Normalisation:* none. The matrix is already log2 and quantile-normalised (3.1), so re-normalising would only distort it.
- *Features:* 43,758 probes → 30,505 with a gene symbol → 19,541 genes (one probe per gene, the shared default) → **17,528 expressed genes**. The 2,013 genes removed are not above the negative-control background (log2 3.044) in at least 20 % of the 221 main samples. The threshold comes from the array's own negative controls, not from the data we want to test, and it ignores time and genotype. Gene counts change modestly with the settings (16,817–17,929 for 10–50 % of samples at this threshold; `qc_filter_sensitivity.csv`), and no clock gene is lost at any threshold up to log2 5. A cut at log2 6, the stub's placeholder, would have removed PER3.
- *Samples:* 438 → **221** (Sleep Extension, 24 subjects: 11 × 4/4, 13 × 5/5). The 217 Sleep Restriction samples are set aside for the sensitivity analysis. **QC removed no sample.**

**Is anything suspicious left?**
- 4 mild correlation outliers (AF0069 T10, AF0114 T1 and T6, AF0145 T4) and 6 arrays with bright control spots. All are kept and listed, and none looks like a failed or swapped array.
- AF0182 SE's inconsistent phase labels are handled by `for_phase_analysis=True`; its arrays are normal.

**Does the data agree with biology that must hold?** Yes.
- All clock genes are expressed above background, with PER3 lowest.
- PER1–3 and NR1D1/2 peak together in the early morning, roughly opposite ARNTL.
- Sex markers are consistent within every subject.

**Do the shared defaults stand?** QC supports them, so none is proposed for change.
- *One probe per gene:* every selected clock-gene probe is well above background. The two unused PER3 probes average 3.9 and 4.2, near background, which confirms that averaging them in would dilute PER3.
- *≥ 6 timepoints:* this rule removes no SE subject.
- *Sample QC:* nothing justifies dropping a sample.

**What QC cannot show.** Expression QC cannot catch a sample whose *time label* is wrong while its RNA is fine (AF0182 was found from its timing, not its expression). Sex is inferred from two genes, not recorded. The background threshold is a judgement, though the sensitivity table shows it does not drive the result.

**Handed to Parts 3–4:**
- `data/processed/expressed_genes_SE.csv`: the expressed-gene set, for the rhythmicity scan's features and as the enrichment background.
- `qc_sample_metrics.csv`: per-sample QC flags.
- PER3 is the noisiest primary gene, so expect wider confidence intervals for it.
